# Project 6: Privacy-Preserving Machine Learning
Bach Nguyen, Son Nguyen

This project explore application of the Differentially Private ID3 algorithm on the Adult Census Income dataset. 

In [78]:
import os
import os.path
import pandas as pd
import numpy as np

## Loading in the data

In [79]:
np.random.seed(42)

datadir = "data"
data = os.path.join(datadir, "adult.data")
df = pd.read_csv(data)
df.columns = ["age", "workclass", "fnlwgt", "education", "education-num", 
              "marital-status", "occupation", "relationship", "race", "sex", 
              "capital-gain", "capital-loss", "hours-per-week", "native-country", "income"]

# Dropping education-num
df.drop(columns=["education-num", "fnlwgt"], inplace=True)

# Drop rows with missing values
df.dropna(inplace=True)




In [80]:
from dp_id3 import DPDecisionTree
from sklearn.model_selection import train_test_split


## Train-Test Split


In [81]:

# Split the data into training and testing sets
X = df.drop(columns=["income"])
y = df["income"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Combine features and target for training (required by DPDecisionTree.fit)
train_df = X_train.copy()
train_df["income"] = y_train

## Training the Decision Tree and Making Predictions


In [82]:
# Initialize and train the differentially private decision tree
tree = DPDecisionTree(epsilon1=1.0, max_depth=6)
tree.fit(train_df, "income")

# Make predictions on the test set
y_pred = tree.predict(X_test)


/Users/admin/Documents/SonNguyen/CS323/cs323-project-6/dp_id3.py:45: RuntimeWarning: invalid value encountered in log2
  return float(-np.sum(p * np.log2(p)))
/Users/admin/Documents/SonNguyen/CS323/cs323-project-6/dp_id3.py:45: RuntimeWarning: invalid value encountered in log2
  return float(-np.sum(p * np.log2(p)))
/Users/admin/Documents/SonNguyen/CS323/cs323-project-6/dp_id3.py:45: RuntimeWarning: invalid value encountered in log2
  return float(-np.sum(p * np.log2(p)))
/Users/admin/Documents/SonNguyen/CS323/cs323-project-6/dp_id3.py:45: RuntimeWarning: invalid value encountered in log2
  return float(-np.sum(p * np.log2(p)))
/Users/admin/Documents/SonNguyen/CS323/cs323-project-6/dp_id3.py:45: RuntimeWarning: invalid value encountered in log2
  return float(-np.sum(p * np.log2(p)))
/Users/admin/Documents/SonNguyen/CS323/cs323-project-6/dp_id3.py:45: RuntimeWarning: invalid value encountered in log2
  return float(-np.sum(p * np.log2(p)))
/Users/admin/Documents/SonNguyen/CS323/cs323-p

## Computing Accuracy


In [83]:
# Calculate accuracy
accuracy = np.mean(y_test.values == y_pred)
print(f"Accuracy: {accuracy:.4f} ({accuracy*100:.2f}%)")


Accuracy: 0.8194 (81.94%)


## Testing Different Parameter Combinations


In [84]:
# Test different parameter combinations
import warnings
warnings.filterwarnings('ignore')  # Suppress log2 warnings for cleaner output

results = []

# Define parameter combinations to test
parameter_combinations = [
    (0.01, 3, "Very High Privacy"),
    (0.1, 4, "High Privacy"),
    (0.5, 5, "Moderate Privacy"),
    (1.0, 6, "Balanced (Default)"),
    (5.0, 8, "Lower Privacy, Higher Accuracy"),
]

print("Training models with different parameters...\n")
print("=" * 70)

for epsilon1, max_depth, description in parameter_combinations:
    print(f"\nTraining: epsilon1={epsilon1}, max_depth={max_depth} ({description})")
    
    # Train model
    tree = DPDecisionTree(epsilon1=epsilon1, max_depth=max_depth)
    tree.fit(train_df, "income")
    
    # Make predictions
    y_pred = tree.predict(X_test)
    
    # Calculate accuracy
    accuracy = np.mean(y_test.values == y_pred)
    
    # Store results
    results.append({
        'epsilon1': epsilon1,
        'max_depth': max_depth,
        'description': description,
        'accuracy': accuracy
    })
    
    print(f"  Accuracy: {accuracy:.4f} ({accuracy*100:.2f}%)")

print("\n" + "=" * 70)


Training models with different parameters...


Training: epsilon1=0.01, max_depth=3 (Very High Privacy)
  Accuracy: 0.7302 (73.02%)

Training: epsilon1=0.1, max_depth=4 (High Privacy)
  Accuracy: 0.7869 (78.69%)

Training: epsilon1=0.5, max_depth=5 (Moderate Privacy)
  Accuracy: 0.8154 (81.54%)

Training: epsilon1=1.0, max_depth=6 (Balanced (Default))
  Accuracy: 0.8146 (81.46%)

Training: epsilon1=5.0, max_depth=8 (Lower Privacy, Higher Accuracy)
  Accuracy: 0.8386 (83.86%)

